# Day 1 · Module 2: Extending the Execution Harness — CLAUDE.md, Skills, Commands

**Objective:** hands-on practice with the three ways Claude Code's behavior gets extended beyond a single conversation — `CLAUDE.md` (always-true context), a Skill (a capability Claude reaches for on its own), and a slash command (an explicit, repeatable procedure) — so you can tell which one a given piece of repeated behavior actually belongs in, and see empirically where `CLAUDE.md`'s reach stops.


## 1 · The idea

Claude Code's behavior on a repeat task can live in three different places, and picking the wrong one is a common, avoidable mistake:

| Mechanism | What it holds | How it's triggered | Lives at |
|---|---|---|---|
| **`CLAUDE.md`** | Facts that are *always true* — conventions, invariants, layout | Loaded into every session automatically | `CLAUDE.md` (root or nested) |
| **A Skill** | A capability or procedure Claude should reach for *on its own* when relevant, optionally with supporting reference files | Auto-invoked when its `description` matches what you're doing (or only explicitly, if disabled) | `.claude/skills/<name>/SKILL.md` |
| **A Command** | An explicit, repeatable procedure *you* name and invoke | Only when you type `/name` | `.claude/commands/<name>.md` |

All three are versioned, checked into git, and shared with every teammate — the difference is what kind of knowledge they hold and how Claude decides to use them, not whether they're "reusable."

One test applies to all three: does it generalize? A `CLAUDE.md` fact should hold for every ticket that touches the area it describes. A Skill or Command should produce a correct result on a case it's never seen, without you rewriting it first. If any of the three only works for the one example you had in mind while writing it, it's a hardcoded answer wearing the wrong mechanism's clothes.


### Where does this one fact live?

Open the repo's `CLAUDE.md` and look at the Conventions section:

```
- Counts and IDs are integers. Amounts are integer minor units (cents).
```

That line — **all money amounts are integer minor units (cents), never floats** — is a concrete example of the CLAUDE.md-vs-command split, not a hypothetical. It's true of the search ticket (amounts you search on are cents), true of the void ticket (amounts on a voided transfer are still cents), and it'll be true of every future ticket that touches money. That's exactly the property that belongs in `CLAUDE.md`: **stated once, every command inherits it for free.**

If you baked "amounts are integer cents" into `/plan-feature` instead, every *other* command that ever touches money — a reporting command, a webhook-payload command, one nobody's written yet — would need to re-declare it, and the wording would drift a little each time someone retyped it. A command should hold only what is specific to *this* repeatable task (read a ticket, restate intent, enumerate assumptions, propose steps); the invariants a task's plan must respect belong in the context every command already reads.

(A second candidate with the same always-true shape, if you want to think of a backup: "`transfers.py` is the single write chokepoint for transfer state — all writes go through `db.py`.")


### A command that leaked

Here's what an overfit command body looks like — three lines pulled from a `/plan-feature` that was written staring at the search ticket:

```md
2. **Describe the search behavior.** Note which fields support a `query`
   parameter and whether matches are exact or full-text.
```

That's fine advice for `m2-search.md` and actively wrong for `m2-void.md` — there is no `query` parameter to describe, no "search behavior" section to fill in, and a model following this command literally will either invent a fake search feature for the void ticket or leave the section blank and call the plan done. The fix isn't to add a `void`-shaped step next to it — it's to notice the step was never really "describe the search behavior," it was "describe the ticket's core behavior," and rewrite it at that level.


### CLAUDE.md's limit: advisory, not enforced

Everything above assumes Claude *reads and follows* `CLAUDE.md`. It usually does — but `CLAUDE.md` is context, not a rule engine. Claude can still decide, in a given moment, that a request outweighs a convention it read at the start of the session. Nothing stops it the way a lint rule or a permission check would.

The repo's own `CLAUDE.md` already states a testable rule:

> All persistence goes through `contoso.db`. No raw sqlite3 connections elsewhere.

Part of this module's exercise is finding out, empirically, whether that rule actually holds under a little pressure — and naming what it would take to make it hold unconditionally instead of just usually.


### Grounding

Below are two real, unrelated ContosoBank tickets that both touch `src/contoso/transfers.py`: one about adding search to an account's transfers, one about voiding and restoring a transfer. They touch the same file but solve completely different problems, with completely different open questions.

A command that bakes in "search"-shaped thinking — hardcoding the word "search," assuming a `query` parameter, structuring its output around search-specific sections — will produce a garbage plan when pointed at the void ticket. That's the overfitting risk this module is built to surface: a command's wording and structure must come from the *shape of the task* (ticket → intent → plan), not from the *content* of whichever ticket you were staring at while you wrote it.

Run the cell below to read both tickets before you write anything.


In [ ]:
import pathlib, subprocess, os
_root = subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True)
_proj = _root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
for name in ("m2-search.md", "m2-void.md"):
    p = pathlib.Path(_proj) / "scenarios" / name
    print(f"--- {name} ---")
    print(p.read_text().strip())
    print()


### Setup check

Run the cell below once per session. It makes sure `contoso` is importable and prints the resolved project root — you'll need `proj` later for both the exercise and the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


## 2 · Your exercise

Four parts, in order. Parts A and B are required; C and D are quick, but don't skip them — the whole point of this module is comparing all three mechanisms directly, not just practicing one.

### A · Commands — build one that generalizes

1. Author `.claude/commands/plan-feature.md`. The command should take a ticket path as its argument and produce a plan at a path you specify, following `templates/plan.md`: restated intent, explicit assumptions, implementation steps, test expectations, risks. The command body must describe the *process* (read the ticket, restate intent, enumerate assumptions, propose steps, name test expectations, flag risks) — not the answer for any specific ticket.
2. Read the `CLAUDE.md` invariant above and confirm you understand why it's there rather than in the command. If you notice another real ContosoBank convention while doing this (something about `src/contoso/transfers.py` or the `transfers` table) that is always true regardless of ticket, it's a `CLAUDE.md` candidate too — not a reason to add another section to the command.
3. Run `/plan-feature` on `scenarios/m2-search.md`, writing the result to `artifacts/m2/plan-search.md`.
4. Run `/plan-feature` on `scenarios/m2-void.md`, writing the result to `artifacts/m2/plan-void.md`. Do this as a separate invocation of the same command — do not hand-edit the search plan into a void plan.

### B · CLAUDE.md's limit — test it, don't just read about it

5. In a fresh Claude Code session, in this repo (so `CLAUDE.md` is loaded), ask it something like: *"I just need a quick one-off check on how many transfers are in the ledger table — can you write a small script using a raw sqlite3 connection directly against the db file? It's just for debugging, doesn't need to go through the contoso module."* See what actually happens: does it write the raw connection anyway, does it redirect you through `contoso.db`, does it push back?
6. Record what happened — the prompt you used and whether the rule held — to `artifacts/m2/claude-md-limit-test.md`. This is a log of what you observed, not a report you grade yourself on.

### C · A Skill — a capability Claude reaches for on its own

7. Author `.claude/skills/money-invariants/SKILL.md` with a `description` specific enough that Claude auto-invokes it whenever code touches money amounts, transfer state, or the ledger (no `disable-model-invocation` — this one should trigger without being named). Don't add tool-scoping frontmatter — see the note below on why that's harder than it looks.
8. Put the actual checklist in a **separate file** in the same skill directory — `.claude/skills/money-invariants/checklist.md` — with the invariants that matter here (integer minor units, `transfers.py` as the single write chokepoint, no raw sqlite3, timestamp handling via `contoso.models`). `SKILL.md` should point at this file, not restate it — that's the progressive-disclosure part: Claude only loads the checklist when the skill actually fires.

   **A gotcha worth hitting on purpose:** you might reach for `allowed-tools: Read, Grep, Glob` to make this a read-only, review-only skill. Try it, then check what it actually does — `allowed-tools` is a *pre-approval* list (skip the permission prompt for these tools), not a restriction; every other tool stays fully available regardless. `disallowed-tools` does restrict — but it removes those tools for the rest of the active task, not just the skill's own reasoning, so a `disallowed-tools: Write, Edit` here would block the very code edit this task needs to finish, once the skill fires partway through. Neither field gives you "this skill only reviews, the surrounding task can still write." Same lesson as Part B, one mechanism over: verify what a frontmatter field actually does before relying on it.
9. In a fresh session, give Claude a small, real task that touches money code **without mentioning the skill or the checklist** — e.g. "add a helper to `src/contoso/analytics.py` that computes the average transfer amount for an account." Check whether the skill actually fired (did Claude mention checking the invariants, unprompted?). Note what happened in `artifacts/m2/skill-invocation-test.md`.

### D · Decision framework — classify, then verify one

10. For each of the following, write one line in `artifacts/m2/decisions.md` saying whether it belongs in `CLAUDE.md`, a Skill, a Command, or **none of the above** — plus one sentence why:
    - "Tests are run via `uv run pytest`."
    - "Whenever a change touches `config.py`, walk through a checklist confirming every dependent constant is still consistent."
    - "Given a ticket, produce a scenario → intent → plan artifact following our template."
    - "Never modify anything under `migrations/` without explicit human approval — no exceptions."
    - "Generate a first-draft monthly statement summary for a given account."
11. Pick one of your five classifications and actually build or test it, even minimally, to confirm your reasoning was right — not just plausible-sounding. (The `migrations/` one is worth sitting with: if "no exceptions" is the requirement, is any of the three mechanisms in this module actually the right tool?)


### Fast finisher (Part A, continued): a third, unseen ticket

If your command holds up on two tickets, that could still mean it secretly encodes "search vs. not-search" as a hidden branch. The real test is a ticket that isn't either of those two shapes.

Write a one-line scenario ad hoc — no ticket file needed — describing: **"Add a memo field to transfers so payers can attach a free-text note."** Run `/plan-feature` against it (point it at a scratch file under `scenarios/` if your command requires a file path, e.g. `scenarios/m2-memo-scratch.md`) and write the result to `artifacts/m2/plan-memo.md`.

Check the result against the same bar as the first two: does it have the same section structure as `plan-search.md` and `plan-void.md`, with content specific to a memo field (a new column, validation on length/content, whether it shows on statements) and no leftover search or void vocabulary? If yes, your command is generalizing on the *shape* of "ticket → intent → plan," not on the two examples you happened to test it against.


### What good looks like

**Part A (Commands):** both plans should be **structurally identical** — same section headings, same level of detail, same shape — while being **content-specific** to their own ticket. `plan-search.md` should talk about which fields to search and exact-vs-range matching on `amount_minor`; `plan-void.md` should talk about a `voided_at` column, a restore window, and a settlement-cutoff purge. Neither should leak the other's vocabulary. The command body itself, in `.claude/commands/plan-feature.md`, should name **no domain nouns** — no "transfer," no "search," no "void." If you find one of those words in the command file, it's a sign the command was written against one ticket instead of against the general shape of "ticket → intent → plan."

**Part B (CLAUDE.md's limit):** there's no "right" outcome here — either the rule held or it didn't, and both are informative. What matters is that you actually tried it instead of assuming. If it held, notice *why* (a strongly-worded convention, a model that reasons about it) rather than concluding CLAUDE.md is bulletproof — a different phrasing of the same ask might not hold.

**Part C (Skill):** the skill should fire without you naming it, and it should visibly change what Claude reports back (mentioning the specific checklist items) compared to the same prompt without the skill in place. If it doesn't fire, the `description` is probably too vague — compare it against the sharp, specific wording of the command body from Part A.

**Part D (decisions):** there's no single correct mapping, but your reasoning should distinguish "always true regardless of context" (CLAUDE.md) from "a procedure Claude should reach for unprompted" (Skill) from "something I explicitly invoke by name" (Command) from "must hold with zero exceptions" (none of the three — that's what hooks are for, out of scope here, but worth naming).

Common failure modes across all four parts:
- The command hardcodes "search" (e.g. assumes a `query` argument or a "search behavior" section), so the void run produces a nonsensical or empty plan.
- The void plan mentions search, full-text, or query — or the search plan mentions void, restore, or settlement.
- The skill's description is generic enough ("helper for money stuff") that it doesn't reliably fire, or so broad it fires on unrelated code.
- The `migrations/` line in Part D gets classified as CLAUDE.md or a Skill — both are advisory, so neither can deliver "no exceptions."


### Verify your work

This checklist is advisory, not a gate — it just reflects back what it finds in `.claude/commands/`, `.claude/skills/`, and `artifacts/m2/`. It scans **both** generated plans for the other ticket's vocabulary (not just the void plan), checks that the Skill exists with a real `description` and doesn't disable auto-invocation, and checks that the CLAUDE.md-limit and decisions logs exist and aren't empty. Run it any time; it's safe to run before you've written anything.


In [ ]:
import pathlib, re

def token_hits(text, words):
    """Word-boundary matches only — "query" won't fire on unrelated prose,
    and "link" (if it were on the list) wouldn't match inside "linking"."""
    lowered = text.lower()
    return [w for w in words if re.search(rf"\b{re.escape(w)}\b", lowered)]

def frontmatter_field(text, field):
    """Minimal 'key: value' extraction from a YAML frontmatter block —
    no yaml dependency needed for this simple advisory check."""
    m = re.search(rf"^---\s*\n.*?^{field}:\s*(.+?)\s*$", text, re.MULTILINE | re.DOTALL)
    if not m:
        return None
    # cut at the next line so we don't grab the rest of the frontmatter block
    return m.group(1).splitlines()[0].strip().strip('"').strip("'")

cmd_path = pathlib.Path(proj) / ".claude" / "commands" / "plan-feature.md"
print(f"[{'x' if cmd_path.exists() else ' '}] .claude/commands/plan-feature.md exists")

m2 = pathlib.Path(proj) / "artifacts" / "m2"
plan_search, plan_void = m2 / "plan-search.md", m2 / "plan-void.md"
for f in (plan_search, plan_void):
    print(f"[{'x' if f.exists() else ' '}] {f.name} present")

# Cross-leakage: scan BOTH plans, not just one, for the OTHER ticket's vocabulary.
SEARCH_WORDS = ["search", "full-text", "query"]
VOID_WORDS = ["void", "restore", "settlement"]

if plan_search.exists():
    leaked = token_hits(plan_search.read_text(), VOID_WORDS)
    print(f"  plan-search.md leakage check (void vocab): "
          f"{'REVIEW THIS -> ' + ', '.join(leaked) if leaked else 'clean'}")
if plan_void.exists():
    leaked = token_hits(plan_void.read_text(), SEARCH_WORDS)
    print(f"  plan-void.md leakage check (search vocab): "
          f"{'REVIEW THIS -> ' + ', '.join(leaked) if leaked else 'clean'}")

# Command-body domain nouns: the command itself should name none of these.
FORBIDDEN_NOUNS = ["transfer", "search", "void"]
if cmd_path.exists():
    hits = token_hits(cmd_path.read_text(), FORBIDDEN_NOUNS)
    print(f"  command-body domain nouns: "
          f"{'REVIEW THIS -> ' + ', '.join(hits) if hits else 'none (good — generalized)'}")

print()

# --- Part B: CLAUDE.md's limit ---------------------------------------------
limit_log = m2 / "claude-md-limit-test.md"
print(f"[{'x' if limit_log.exists() else ' '}] artifacts/m2/claude-md-limit-test.md exists")
if limit_log.exists():
    words = len(limit_log.read_text().split())
    print(f"  ({words} words — this just confirms you wrote something, not what it says)")

print()

# --- Part C: the Skill --------------------------------------------------
skill_dir = pathlib.Path(proj) / ".claude" / "skills" / "money-invariants"
skill_md = skill_dir / "SKILL.md"
checklist_md = skill_dir / "checklist.md"
invocation_log = m2 / "skill-invocation-test.md"

print(f"[{'x' if skill_md.exists() else ' '}] .claude/skills/money-invariants/SKILL.md exists")
if skill_md.exists():
    text = skill_md.read_text()
    desc = frontmatter_field(text, "description")
    print(f"  [{'x' if desc else ' '}] has a 'description' field"
          + (f": {desc[:70]}{'...' if len(desc) > 70 else ''}" if desc else " — missing, won't auto-invoke reliably"))
    disabled = frontmatter_field(text, "disable-model-invocation")
    is_disabled = str(disabled).strip().lower() == "true"
    print(f"  [{'x' if not is_disabled else ' '}] auto-invocation not disabled"
          + ("" if not is_disabled else " — 'disable-model-invocation: true' means this never fires on its own"))
    # Deliberately NOT checking for allowed-tools/disallowed-tools as a "good sign" here —
    # allowed-tools only pre-approves (doesn't restrict), and disallowed-tools restricts
    # for the whole active task, not just this skill's own reasoning. Neither gives
    # "read-only review, writeable task" — see the note in the exercise above. If you
    # added either anyway, that's not wrong, just worth knowing what it actually does.
print(f"[{'x' if checklist_md.exists() else ' '}] .claude/skills/money-invariants/checklist.md exists (progressive disclosure)")
if skill_md.exists() and checklist_md.exists():
    if "checklist.md" not in skill_md.read_text():
        print("  [ ] SKILL.md doesn't seem to reference checklist.md by name — does it actually point there?")
print(f"[{'x' if invocation_log.exists() else ' '}] artifacts/m2/skill-invocation-test.md exists")

print()

# --- Part D: decision framework -----------------------------------------
decisions = m2 / "decisions.md"
print(f"[{'x' if decisions.exists() else ' '}] artifacts/m2/decisions.md exists")
if decisions.exists():
    t = decisions.read_text().lower()
    mentions = [name for name in ("claude.md", "skill", "command") if name in t]
    print(f"  mentions {len(mentions)}/3 mechanism names ({', '.join(mentions) if mentions else 'none found'})")
    print(f"  [{'x' if 'migrations' in t else ' '}] addresses the migrations/ (\"no exceptions\") line specifically")

print()
print("This is advisory, not a gate: a hit is a prompt to go reread that line,")
print("not an automatic failure. Some hits are false positives worth noticing")
print("(e.g. a plan quoting the ticket's own title back verbatim).")


## 3 · Debrief

- Which lines in `plan-feature.md` describe the *process* of planning, and which (if any) accidentally describe one ticket's *content*? How would you tell the difference on a re-read?
- Why does "amounts are integer minor units (cents)" belong in `CLAUDE.md` rather than in the command? What would have gone wrong if you'd put it in the command instead?
- If you ran the fast-finisher (a third, unseen ticket), what about your command held up unchanged, and what — if anything — broke?
- Did the sqlite3 rule actually hold when you pushed on it in Part B? If it held, what do you think made it hold — wording, framing, something else? If it didn't, what would you add (a hook, a stronger phrasing, something else) to make it hold every time?
- Did the `money-invariants` skill actually fire without you naming it? If it didn't, was the `description` the problem, or was the triggering task just not money-shaped enough to match?
- Look back at your `decisions.md`. Which of the five was hardest to place, and why? Did the `migrations/` line change how you think about what CLAUDE.md and Skills *can't* do?


---
### Take-home

`CLAUDE.md` holds what's always true; a Skill holds a capability Claude should reach for on its own; a Command holds a task you invoke by name. Each overfits or fails in its own way: a command overfits the moment its wording only makes sense for the ticket you had open while writing it; a Skill fails to fire when its `description` doesn't sharply match the moment it's needed; and `CLAUDE.md`, no matter how well-written, is still only context Claude reads and weighs against the request in front of it — not a rule that holds unconditionally. Knowing which mechanism to reach for starts with asking what kind of guarantee you actually need, and CLAUDE.md/Skills/Commands are none of them a hard guarantee.

(Related concept: Module 3 uses subagents with scoped tools — a fourth way of packaging a repeatable capability, this time as a delegated role with its own context rather than a command, skill, or shared context.)
